In [2]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
# SETUP 

TRAINING_DATA = "training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id', 
    'best_tasks_free', 
    'acad_tasks_rating', 
    'best_tasks_select', 
    'subopt_freq_rating',  
    'subopt_tasks_select', 
    'subopt_tasks_free', 
    'evidence_freq_rating', 
    'verify_freq_rating', 
    'verify_method_free', 
    'target'
    ]

text_cols = [
    'best_tasks_free', 
    'subopt_tasks_free', 
    'verify_method_free',
    ] 


ord_cols = [
    'acad_tasks_rating', 
    'subopt_freq_rating',  
    'evidence_freq_rating', 
    'verify_freq_rating', 
    ]

acad_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories_per_col = {
    'acad_tasks_rating': acad_categories,
    'subopt_freq_rating': freq_categories,
    'evidence_freq_rating': freq_categories,
    'verify_freq_rating': freq_categories,
}

cat_cols = [
    'best_tasks_select', 
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [4]:
# EXPLORE DATA 

df.isnull().sum()

id                       0
best_tasks_free          2
acad_tasks_rating        0
best_tasks_select       15
subopt_freq_rating       1
subopt_tasks_select     22
subopt_tasks_free       11
evidence_freq_rating    62
verify_freq_rating       4
verify_method_free      10
target                   0
dtype: int64

In [ ]:
# """These are EQUIVALENT:

# Option 1: make_pipeline (shorter)
# text_pipeline = make_pipeline(
#     SimpleImputer(strategy="constant", fill_value=""),
#     TfidfVectorizer(max_features=2000)
# )

# Option 2: Pipeline (explicit)
# text_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy="constant", fill_value="")),
#     ('tfidf', TfidfVectorizer(max_features=2000))
# ])"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# custom function makecorpus just to keep consistency in pipeline 
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    # TFIDF manually (classical)
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # hardcoded stop_words param for now 
        # TODO: try with max instead 
        ('tfidf', TfidfVectorizer(stop_words='english'))
    ])
      
def preprocess_ord(): 
    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories= [acad_categories] * len(ord_cols)),
    # TODO: encode the dimension of the overall feature vector, do KNN on that, euclidian disatnce between datapoints  
    # SimpleImputer(strategy="constant", fill_value="most_frequent", missing_values=np.nan),
    )
    
def preprocess_cat():
    pass 
    # TODO: research how to preprocess these
    
def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem 
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ] 
    
    return ColumnTransformer(transformers=transformers)

# Full pipeline 
# added logistic regression as placeholder classifier for now 
preprocessor = create_preprocessor()
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# ============ GRID SEARCH ON FULL PIPELINE ============

param_grid = {
    # Text TF-IDF parameters
    'preprocessor__text__tfidf__max_features': [500, 1000, 2000, 3000],
    'preprocessor__text__tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'preprocessor__text__tfidf__min_df': [1, 2, 3, 4, 5],
    'preprocessor__text__tfidf__max_df': [0.8, 0.9, 1.0],
    
    # Classifier parameters
    # TODO: change to SVM or RandomForest and tune their hyperparams
}

# Grid search
grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit on raw data (pipeline handles preprocessing)
X = df[text_cols + ord_cols + cat_cols]
y = df['target']

# # df specified here 
# grid_search.fit(X, y)

# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

In [ ]:
# Implementation of TFIDF

import re
from collections import Counter

STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "at",
    "with", "is", "it", "this", "that", "as", "by", "be", "are", "was",
    "were", "from", "but", "if", "so", "not", "isn", "t"
}

TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)

def tokenize(text, lower=True, remove_stopwords=True, stopwords=STOPWORDS):
    if not isinstance(text, str):
        return []
    if lower:
        text = text.lower()
    tokens = TOKEN_PATTERN.findall(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

def generate_ngrams(tokens, ngram_range=(1, 1)):
    min_n, max_n = ngram_range
    L = len(tokens)
    all_ngrams = []
    for n in range(min_n, max_n + 1):
        if L < n:
            continue
        for i in range(L - n + 1):
            all_ngrams.append(" ".join(tokens[i:i+n]))
    return all_ngrams

def fit_tfidf(
    corpus,
    ngram_range=(1, 3),
    min_df=1,
    max_df=1.0,
    max_features=None,
    remove_stopwords=True,
):
    """
    TF-IDF formula:
      - tokenization + n-grams
      - df filtering with min_df / max_df
      - idf(t) = log((1 + N) / (1 + df)) + 1
      - L2-normalization per document
    """
    N = len(corpus)
    doc_terms = []
    df_counter = Counter()

    for doc in corpus:
        tokens = tokenize(doc, remove_stopwords=remove_stopwords)
        terms = generate_ngrams(tokens, ngram_range=ngram_range)
        doc_terms.append(terms)

        unique_terms = set(terms)
        for t in unique_terms:
            df_counter[t] += 1

    vocab_terms = []
    for term, df in df_counter.items():
        if df < min_df:
            continue
        if isinstance(max_df, float) and max_df <= 1.0:
            if df > max_df * N:
                continue
        else:
            if df > max_df:
                continue
        vocab_terms.append(term)

    if max_features is not None:
        vocab_terms = sorted(
            vocab_terms,
            key=lambda t: df_counter[t],
            reverse=True
        )[:max_features]
    else:
        vocab_terms = sorted(vocab_terms)

    vocab = {term: j for j, term in enumerate(vocab_terms)}
    V = len(vocab)

    idf = np.zeros(V, dtype=np.float64)
    for term, j in vocab.items():
        df = df_counter[term]
        idf[j] = np.log((1.0 + N) / (1.0 + df)) + 1.0

    X_tfidf = np.zeros((N, V), dtype=np.float64)
    for i, terms in enumerate(doc_terms):
        if not terms:
            continue
        tf_counts = Counter(t for t in terms if t in vocab)
        if not tf_counts:
            continue
        doc_len = sum(tf_counts.values())
        for term, cnt in tf_counts.items():
            j = vocab[term]
            tf = cnt / doc_len
            X_tfidf[i, j] = tf * idf[j]

    norms = np.linalg.norm(X_tfidf, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    X_tfidf = X_tfidf / norms

    return X_tfidf, vocab, idf


In [ ]:
# Ordinal data preprocessing

from collections import Counter
import numpy as np
import pandas as pd

# Create mappings and fill values(frequency based)
def fit_ordinal_encoders(df, ord_cols, ord_categories_per_col):
    mappings = {}
    fill_values = {}

    for col in ord_cols:
        cats = ord_categories_per_col[col]
        mapping = {cat: i for i, cat in enumerate(cats)}
        mappings[col] = mapping

        col_vals = df[col].dropna()
        counts = Counter(v for v in col_vals if v in mapping)

        if counts:
            most_common_cat, _ = counts.most_common(1)[0]
            fill_values[col] = mapping[most_common_cat]
        else:
            # fallback: middle category index if column is weird/empty
            fill_values[col] = len(cats) // 2

    return mappings, fill_values

# Trnasform data from dataset using mappings
def transform_ordinal(df, ord_cols, mappings, fill_values):
    N = len(df)
    K = len(ord_cols)
    X_ord = np.zeros((N, K), dtype=np.float64)

    for j, col in enumerate(ord_cols):
        mapping = mappings[col]
        fill_val = fill_values[col]

        encoded_col = []
        for v in df[col].tolist():
            if pd.isna(v) or v not in mapping:
                encoded_col.append(fill_val)
            else:
                encoded_col.append(mapping[v])

        X_ord[:, j] = np.array(encoded_col, dtype=np.float64)

    return X_ord

In [ ]:
# Categorical preprocessing(multi select)

# parse text from multiselect questions to get categories selected
def parse_multiselect(raw_value, choice_list):
    if pd.isna(raw_value):
        return []  # skipped → no selections

    txt = str(raw_value).strip()
    if txt == "":
        return []

    selections = []
    for choice in choice_list:
        if choice in txt:
            selections.append(choice)

    return selections

# Transform categorical data to one-hot encoding for mult-select categorical data
def transform_multiselect_categorical(df, cat_cols, cat_multi_select_choices):
    N = len(df)
    M = len(cat_multi_select_choices)

    # total features = (#cat_cols) * (#choices)
    X = np.zeros((N, len(cat_cols) * M), dtype=np.float32)
    feature_names = []

    # build feature name list in the same order we fill X
    for col in cat_cols:
        for choice in cat_multi_select_choices:
            feature_names.append(f"{col}__{choice}")

    for i in range(N):
        offset = 0
        for col in cat_cols:
            selections = parse_multiselect(df.loc[i, col], cat_multi_select_choices)
            for j, choice in enumerate(cat_multi_select_choices):
                X[i, offset + j] = 1.0 if choice in selections else 0.0
            offset += M

    return X, feature_names


In [ ]:
# fill for empty text responses
df[text_cols] = df[text_cols].fillna("")

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus'].tolist()
labels = df['target'].to_numpy()

# TFIDF params(currently retrieved from best params from sklearn using log reg)
ngram_range = (1, 3)
min_df = 1
max_df = 1.0
max_features = None

X_tfidf, vocab, idf = fit_tfidf(
    corpus,
    ngram_range=ngram_range,
    min_df=min_df,
    max_df=max_df,
    max_features=max_features,
    remove_stopwords=True,
)

print("Manual TF-IDF matrix shape:", X_tfidf.shape)
print("Vocabulary size:", len(vocab))

ord_mappings, ord_fill_values = fit_ordinal_encoders(df, ord_cols, ord_categories_per_col)
X_ord = transform_ordinal(df, ord_cols, ord_mappings, ord_fill_values)

print("Ordinal feature matrix shape:", X_ord.shape)

X_cat, cat_feature_names = transform_multiselect_categorical(
    df,
    cat_cols=cat_cols,
    cat_multi_select_choices=cat_multi_select_choices,
)

print("Categorical (multi-select) feature matrix shape:", X_cat.shape)
print("Number of categorical features:", len(cat_feature_names))

X_all = np.hstack([X_tfidf, X_ord, X_cat])
print("Final combined feature matrix shape:", X_all.shape)


Manual TF-IDF matrix shape: (825, 66800)
Vocabulary size: 66800
Ordinal feature matrix shape: (825, 4)
Categorical (multi-select) feature matrix shape: (825, 16)
Number of categorical features: 16
Final combined feature matrix shape: (825, 66820)


In [21]:
# Quick test using all features and sklearn log reg

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_all, labels, test_size=0.2, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")    

Train accuracy: 0.859
Test accuracy: 0.655


In [ ]:
#TFIDF sanity check

import numpy as np

print("Manual TF-IDF matrix shape:", X_tfidf.shape)
print("Vocabulary size:", len(vocab))

print("Any NaNs in X_tfidf?", np.isnan(X_tfidf).any())

row_norms = np.linalg.norm(X_tfidf, axis=1)
print("Min row norm:", row_norms.min())
print("Max row norm:", row_norms.max())
print("Number of all-zero rows:", np.sum(row_norms == 0))

i = 0  # or any row index you care about

row = X_tfidf[i]
nonzero_indices = np.where(row > 0)[0]

# get top 20 terms by TF-IDF weight
top_idx = nonzero_indices[np.argsort(row[nonzero_indices])[-20:]][::-1]

inv_vocab = {j: term for term, j in vocab.items()}

print("Document:", df['corpus'].iloc[i][:300], "...")
print("\nTop TF-IDF terms (manual):")
for j in top_idx:
    print(f"{inv_vocab[j]:40s}  {row[j]:.4f}")

Manual TF-IDF matrix shape: (825, 66800)
Vocabulary size: 66800
Any NaNs in X_tfidf? False
Min row norm: 0.0
Max row norm: 1.0000000000000002
Number of all-zero rows: 1


In [23]:
# Manual TFIDF comparision against sklearn

from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    min_df=1,
    max_df=1.0,
    max_features=None,
    norm='l2'
)

X_sklearn = vec.fit_transform(corpus)   # scipy sparse matrix
print("Sklearn TF-IDF shape:", X_sklearn.shape)

# sklearn vocab: term -> index
sk_vocab = vec.vocabulary_

# intersection of terms
common_terms = set(vocab.keys()) & set(sk_vocab.keys())
print("Common terms:", len(common_terms))

# build mappings of indices for common terms
manual_idx = np.array([vocab[t] for t in common_terms])
sk_idx = np.array([sk_vocab[t] for t in common_terms])

# convert X_sklearn to dense *for a subset* to avoid memory blow-up
X_sklearn_dense = X_sklearn.toarray()

# extract aligned matrices
X_man_common = X_tfidf[:, manual_idx]
X_sk_common = X_sklearn_dense[:, sk_idx]

# compute cosine similarity between manual and sklearn vectors for a few docs
def cosine_sim(a, b):
    num = (a * b).sum()
    den = (np.linalg.norm(a) * np.linalg.norm(b))
    return 0.0 if den == 0 else num / den

for i in range(5):  # first 5 docs
    sim = cosine_sim(X_man_common[i], X_sk_common[i])
    print(f"Doc {i} cosine similarity (manual vs sklearn on common vocab): {sim:.4f}")

i = 0

# manual
row_man = X_tfidf[i]
idx_man = np.argsort(row_man)[-100:][::-1]
inv_vocab = {j: term for term, j in vocab.items()}
top_man = [(inv_vocab[j], row_man[j]) for j in idx_man]

# sklearn
row_sk = X_sklearn_dense[i]
idx_sk = np.argsort(row_sk)[-100:][::-1]
inv_sk_vocab = {j: term for term, j in sk_vocab.items()}
top_sk = [(inv_sk_vocab[j], row_sk[j]) for j in idx_sk]

print("Top manual TF-IDF terms:")
for t, w in top_man:
    print(f"{t:40s} {w:.4f}")

print("\nTop sklearn TF-IDF terms:")
for t, w in top_sk:
    print(f"{t:40s} {w:.4f}")


Sklearn TF-IDF shape: (825, 51693)
Common terms: 31730
Doc 0 cosine similarity (manual vs sklearn on common vocab): 0.9921
Doc 1 cosine similarity (manual vs sklearn on common vocab): 0.9770
Doc 2 cosine similarity (manual vs sklearn on common vocab): 0.9086
Doc 3 cosine similarity (manual vs sklearn on common vocab): 0.9926
Doc 4 cosine similarity (manual vs sklearn on common vocab): 1.0000
Top manual TF-IDF terms:
down                                     0.1093
topic                                    0.1002
searches find                            0.0890
text rewording text                      0.0890
forums where                             0.0890
text rewording                           0.0890
find forums where                        0.0890
much watering                            0.0890
find forums                              0.0890
people discuss topic                     0.0890
relatively obvious trivial               0.0890
trivial i double                         0.0890
wate

In [ ]:
# ordinal sanity check

print("Ordinal categories per column:")
for col in ord_cols:
    print(f"{col}: {ord_categories_per_col[col]}")
print()

print("Per-column mappings:")
for col in ord_cols:
    print(f"{col} -> {ord_mappings[col]}")
print()

print("Per-column fill values (index used for NaN/unknown):")
for col in ord_cols:
    idx = ord_fill_values[col]
    print(f"{col}: fill index = {idx} "
          f"(category = {ord_categories_per_col[col][idx]})")
print()

# Show encoded columns alongside original values
ord_df_debug = df[ord_cols].copy()
for j, col in enumerate(ord_cols):
    ord_df_debug[col + "_encoded"] = X_ord[:, j].astype(int)

ord_df_debug.head(10)



Ordinal categories per column:
acad_tasks_rating: ['1 — Not at all likely', '2 — Unlikely', '3 — Neutral / Unsure', '4 — Likely', '5 — Very likely']
subopt_freq_rating: ['1 — Never', '2 — Rarely', '3 — Sometimes', '4 — Often', '5 — Very often']
evidence_freq_rating: ['1 — Never', '2 — Rarely', '3 — Sometimes', '4 — Often', '5 — Very often']
verify_freq_rating: ['1 — Never', '2 — Rarely', '3 — Sometimes', '4 — Often', '5 — Very often']

Per-column mappings:
acad_tasks_rating -> {'1 — Not at all likely': 0, '2 — Unlikely': 1, '3 — Neutral / Unsure': 2, '4 — Likely': 3, '5 — Very likely': 4}
subopt_freq_rating -> {'1 — Never': 0, '2 — Rarely': 1, '3 — Sometimes': 2, '4 — Often': 3, '5 — Very often': 4}
evidence_freq_rating -> {'1 — Never': 0, '2 — Rarely': 1, '3 — Sometimes': 2, '4 — Often': 3, '5 — Very often': 4}
verify_freq_rating -> {'1 — Never': 0, '2 — Rarely': 1, '3 — Sometimes': 2, '4 — Often': 3, '5 — Very often': 4}

Per-column fill values (index used for NaN/unknown):
acad_task

,acad_tasks_rating,subopt_freq_rating,evidence_freq_rating,verify_freq_rating,acad_tasks_rating_encoded,subopt_freq_rating_encoded,evidence_freq_rating_encoded,verify_freq_rating_encoded
0,3 — Neutral / Unsure,3 — Sometimes,1 — Never,4 — Often,2,2,0,3
1,4 — Likely,3 — Sometimes,1 — Never,5 — Very often,3,2,0,4
2,3 — Neutral / Unsure,4 — Often,2 — Rarely,3 — Sometimes,2,3,1,2
3,5 — Very likely,4 — Often,3 — Sometimes,2 — Rarely,4,3,2,1
4,4 — Likely,4 — Often,2 — Rarely,1 — Never,3,3,1,0
5,1 — Not at all likely,1 — Never,1 — Never,1 — Never,0,0,0,0
6,3 — Neutral / Unsure,4 — Often,2 — Rarely,5 — Very often,2,3,1,4
7,1 — Not at all likely,1 — Never,NaN,1 — Never,0,0,2,0
8,1 — Not at all likely,1 — Never,1 — Never,NaN,0,0,0,3
9,3 — Neutral / Unsure,4 — Often,2 — Rarely,5 — Very often,2,3,1,4


In [25]:
# categorical sanity check

print("X_cat shape:", X_cat.shape)
print("Number of categorical features:", len(cat_feature_names))
print("First few feature names:")
for name in cat_feature_names[:10]:
    print("  ", name)

# Pick a small slice of rows for inspection
rows_to_check = range(0, 10)  # first 10 rows

cat_debug = df.loc[rows_to_check, cat_cols].copy()

# Add encoded columns for each feature
for j, feat_name in enumerate(cat_feature_names):
    col = f"cat_{j}"
    cat_debug[col] = X_cat[rows_to_check, j].astype(int)

cat_debug


X_cat shape: (825, 16)
Number of categorical features: 16
First few feature names:
   best_tasks_select__Explaining complex concepts simply
   best_tasks_select__Drafting professional text (e.g., emails, résumés)
   best_tasks_select__Brainstorming or generating creative ideas
   best_tasks_select__Writing or editing essays/reports
   best_tasks_select__Converting content between formats (e.g., LaTeX)
   best_tasks_select__Writing or debugging code
   best_tasks_select__Data processing or analysis
   best_tasks_select__Math computations
   subopt_tasks_select__Explaining complex concepts simply
   subopt_tasks_select__Drafting professional text (e.g., emails, résumés)


,best_tasks_select,subopt_tasks_select,cat_0,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,cat_7,cat_8,cat_9,cat_10,cat_11,cat_12,cat_13,cat_14,cat_15
0,"Math computations,Data processing or analysis,...","Writing or debugging code,Explaining complex c...",1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0
1,Writing or debugging code,"Math computations,Explaining complex concepts ...",0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,1
2,"Math computations,Converting content between f...","Data processing or analysis,Drafting professio...",0,0,0,0,1,0,0,1,0,1,1,0,0,0,1,0
3,"Explaining complex concepts simply,Converting ...","Writing or debugging code,Data processing or a...",1,1,1,1,1,0,0,0,0,1,0,1,0,1,1,0
4,"Math computations,Writing or debugging code,Da...","Data processing or analysis,Explaining complex...",0,0,0,0,1,1,1,1,1,1,1,1,1,0,1,0
5,Brainstorming or generating creative ideas,"Math computations,Writing or debugging code,Da...",0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1
6,"Writing or debugging code,Data processing or a...","Math computations,Writing or debugging code,Co...",1,1,1,1,1,1,1,0,0,0,1,0,1,1,0,1
7,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,"Explaining complex concepts simply,Converting ...","Math computations,Writing or debugging code,Da...",1,0,1,0,1,0,0,0,0,0,0,0,0,1,1,1


In [ ]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude 

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE 
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features 
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range 
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")

In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1) 
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)
     
print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:

def split_data(df):
    students = df['id'].unique()
    train_df, test_df = train_test_split(students, test_size=0.3, random_state=42)
